# IT549: Deep Learning – Lab 3
## Image-Based AQI Classification using CNN and Pretrained Models
**Name:** *(Your Name)*  
**ID:** 202511016

## 📦 Install & Import Libraries

In [ ]:
# Install any missing packages (run once if needed)
# !pip install torch torchvision scikit-learn matplotlib seaborn tqdm pillow pandas

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## Task 1 – Data Preparation

In [ ]:
# ── Paths – update IMAGE_DIR to point at your sampled_images folder ───────────
CSV_PATH   = 'data.csv'          # path to data.csv
IMAGE_DIR  = 'sampled_images'    # folder containing the images

df = pd.read_csv(CSV_PATH)

# Keep only required columns
df = df[['Filename', 'AQI_Class']].rename(columns={'Filename': 'image_path'})

# Encode labels
classes   = sorted(df['AQI_Class'].unique())
class2idx = {c: i for i, c in enumerate(classes)}
idx2class = {i: c for c, i in class2idx.items()}
df['label'] = df['AQI_Class'].map(class2idx)

print(f'Total samples : {len(df)}')
print(f'Classes ({len(classes)}):', classes)
df['AQI_Class'].value_counts()

In [ ]:
# ── Train / Val / Test split  (70 / 15 / 15) ─────────────────────────────────
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['label'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')

In [ ]:
# ── Image transforms ─────────────────────────────────────────────────────────
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

print('Transforms defined.')

In [ ]:
# ── Custom Dataset ────────────────────────────────────────────────────────────
class AQIDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df        = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.image_dir, row['image_path'])
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, int(row['label'])


BATCH = 32

train_dataset = AQIDataset(train_df, IMAGE_DIR, train_transform)
val_dataset   = AQIDataset(val_df,   IMAGE_DIR, val_test_transform)
test_dataset  = AQIDataset(test_df,  IMAGE_DIR, val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH, shuffle=False, num_workers=2)

print(f'Batches – train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}')

In [ ]:
# ── Visualise sample images ───────────────────────────────────────────────────
inv_norm = transforms.Compose([
    transforms.Normalize(mean=[0, 0, 0], std=[1/0.229, 1/0.224, 1/0.225]),
    transforms.Normalize(mean=[-0.485, -0.456, -0.406], std=[1, 1, 1]),
])

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
indices = random.sample(range(len(train_dataset)), 8)
for ax, i in zip(axes.flat, indices):
    img_tensor, label = train_dataset[i]
    img = inv_norm(img_tensor).permute(1, 2, 0).clamp(0, 1).numpy()
    ax.imshow(img)
    ax.set_title(idx2class[label].replace('_', ' '), fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Task 2 – Basic CNN Model (from scratch)

In [ ]:
class BasicCNN(nn.Module):
    """Simple 4-block CNN trained from scratch."""
    def __init__(self, num_classes=6):
        super(BasicCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),           # 112x112
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),           # 56x56
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),           # 28x28
            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2),           # 14x14
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


basic_cnn = BasicCNN(num_classes=len(classes)).to(DEVICE)
print(basic_cnn)
total_params = sum(p.numel() for p in basic_cnn.parameters())
print(f'\nTotal parameters: {total_params:,}')

## Task 3 – Pretrained CNN (Transfer Learning — ResNet-18)

In [ ]:
def build_resnet(num_classes, freeze_backbone=False):
    """Load ImageNet-pretrained ResNet-18 and replace the head."""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    # Replace final fully-connected layer
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


resnet_model = build_resnet(num_classes=len(classes), freeze_backbone=False).to(DEVICE)
total_params = sum(p.numel() for p in resnet_model.parameters())
print(f'ResNet-18 total parameters: {total_params:,}')
print('Final FC layer:', resnet_model.fc)

## Task 4 – Training and Evaluation Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc='Train', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += images.size(0)
    return total_loss / total, correct / total


def full_train(model, train_loader, val_loader, epochs, lr, model_name='model'):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(f'[{model_name}] Epoch {epoch:02d}/{epochs}  '
              f'Train Loss: {tr_loss:.4f}  Train Acc: {tr_acc:.4f}  '
              f'Val Loss: {vl_loss:.4f}  Val Acc: {vl_acc:.4f}')

    # restore best weights
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    print(f'\n✅  Best Val Acc: {best_val_acc:.4f}')
    return history


print('Utility functions defined.')

In [ ]:
# ── Train Basic CNN ───────────────────────────────────────────────────────────
EPOCHS_CNN = 15
LR_CNN     = 1e-3

print('=' * 60)
print('Training Basic CNN from scratch')
print('=' * 60)

history_cnn = full_train(
    basic_cnn, train_loader, val_loader,
    epochs=EPOCHS_CNN, lr=LR_CNN, model_name='BasicCNN'
)

In [ ]:
# ── Train Pretrained ResNet-18 ────────────────────────────────────────────────
EPOCHS_RN = 15
LR_RN     = 1e-4   # lower lr for fine-tuning

print('=' * 60)
print('Fine-tuning ResNet-18')
print('=' * 60)

history_rn = full_train(
    resnet_model, train_loader, val_loader,
    epochs=EPOCHS_RN, lr=LR_RN, model_name='ResNet-18'
)

## Task 5 – Training Curves

In [ ]:
def plot_curves(hist_cnn, hist_rn, epochs):
    ep = range(1, epochs + 1)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Loss
    axes[0, 0].plot(ep, hist_cnn['train_loss'], label='CNN Train')
    axes[0, 0].plot(ep, hist_cnn['val_loss'],   label='CNN Val', linestyle='--')
    axes[0, 0].plot(ep, hist_rn['train_loss'],  label='ResNet Train')
    axes[0, 0].plot(ep, hist_rn['val_loss'],    label='ResNet Val',  linestyle='--')
    axes[0, 0].set_title('Training Loss'); axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss'); axes[0, 0].legend(); axes[0, 0].grid(True)

    # Accuracy
    axes[0, 1].plot(ep, hist_cnn['train_acc'], label='CNN Train')
    axes[0, 1].plot(ep, hist_cnn['val_acc'],   label='CNN Val',    linestyle='--')
    axes[0, 1].plot(ep, hist_rn['train_acc'],  label='ResNet Train')
    axes[0, 1].plot(ep, hist_rn['val_acc'],    label='ResNet Val', linestyle='--')
    axes[0, 1].set_title('Training Accuracy'); axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy'); axes[0, 1].legend(); axes[0, 1].grid(True)

    # CNN only
    axes[1, 0].plot(ep, hist_cnn['train_loss'], label='Train Loss')
    axes[1, 0].plot(ep, hist_cnn['val_loss'],   label='Val Loss', linestyle='--')
    ax2 = axes[1, 0].twinx()
    ax2.plot(ep, hist_cnn['train_acc'], color='green', label='Train Acc', alpha=0.5)
    ax2.plot(ep, hist_cnn['val_acc'],   color='lime',  label='Val Acc',   alpha=0.5, linestyle='--')
    axes[1, 0].set_title('Basic CNN – Loss & Accuracy')
    axes[1, 0].set_xlabel('Epoch'); axes[1, 0].legend(loc='upper left'); ax2.legend(loc='upper right')

    # ResNet only
    axes[1, 1].plot(ep, hist_rn['train_loss'], label='Train Loss')
    axes[1, 1].plot(ep, hist_rn['val_loss'],   label='Val Loss', linestyle='--')
    ax3 = axes[1, 1].twinx()
    ax3.plot(ep, hist_rn['train_acc'], color='darkorange', label='Train Acc', alpha=0.5)
    ax3.plot(ep, hist_rn['val_acc'],   color='gold',       label='Val Acc',   alpha=0.5, linestyle='--')
    axes[1, 1].set_title('ResNet-18 – Loss & Accuracy')
    axes[1, 1].set_xlabel('Epoch'); axes[1, 1].legend(loc='upper left'); ax3.legend(loc='upper right')

    plt.suptitle('Training Curves', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()


plot_curves(history_cnn, history_rn, EPOCHS_CNN)

### Discussion: Why do pretrained models outperform scratch-trained CNNs?

Pretrained models like ResNet-18 have been trained on ImageNet (1.2 million images, 1000 classes). 
Their lower layers already encode **general visual features** — edges, textures, shapes — that are 
universally useful for any image task. Fine-tuning only adapts these rich representations to the 
specific AQI domain, which requires far fewer labelled examples and fewer epochs to converge. 
A CNN trained from scratch must learn all of these features from only ~4200 images, leading to 
slower convergence and higher risk of overfitting. The gap is especially visible in the early epochs 
of the training curves above — ResNet-18 starts with strong accuracy because the backbone weights are 
already meaningful, while the basic CNN starts from random initialisation.

## Task 4 (continued) – Test Set Evaluation & Metrics

In [ ]:
def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            preds  = model(images).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)


def report_metrics(y_true, y_pred, model_name):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    print(f'\n{"="*50}')
    print(f'  {model_name} – Test Results')
    print(f'{"="*50}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}  (weighted)')
    print(f'  Recall    : {rec:.4f}  (weighted)')
    print(f'  F1-Score  : {f1:.4f}  (weighted)')
    print()
    print(classification_report(y_true, y_pred,
                                target_names=[idx2class[i] for i in range(len(classes))]))
    return acc, prec, rec, f1


y_true_cnn, y_pred_cnn = get_predictions(basic_cnn,    test_loader)
y_true_rn,  y_pred_rn  = get_predictions(resnet_model, test_loader)

acc_cnn, prec_cnn, rec_cnn, f1_cnn = report_metrics(y_true_cnn, y_pred_cnn, 'Basic CNN')
acc_rn,  prec_rn,  rec_rn,  f1_rn  = report_metrics(y_true_rn,  y_pred_rn,  'ResNet-18')

In [ ]:
# ── Side-by-side metric bar chart ─────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
cnn_vals = [acc_cnn, prec_cnn, rec_cnn, f1_cnn]
rn_vals  = [acc_rn,  prec_rn,  rec_rn,  f1_rn]

x    = np.arange(len(metrics))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, cnn_vals, width, label='Basic CNN',  color='steelblue')
bars2 = ax.bar(x + width/2, rn_vals,  width, label='ResNet-18', color='darkorange')
ax.set_ylabel('Score'); ax.set_title('Model Comparison on Test Set')
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05); ax.legend()
for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrices ────────────────────────────────────────────────────────
label_names = [idx2class[i].replace('_', '\n') for i in range(len(classes))]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, y_true, y_pred, title in [
    (axes[0], y_true_cnn, y_pred_cnn, 'Basic CNN'),
    (axes[1], y_true_rn,  y_pred_rn,  'ResNet-18'),
]:
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=label_names, yticklabels=label_names)
    ax.set_title(f'Confusion Matrix – {title}', fontsize=13)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices (Test Set)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Task 6 – Misclassification Analysis

In [ ]:
def collect_misclassified(model, dataset, loader, n=10):
    """Return up to n misclassified (image_tensor, true_label, pred_label) tuples."""
    model.eval()
    misclassified = []
    idx = 0
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(DEVICE))
            preds   = outputs.argmax(1).cpu()
            for i in range(len(labels)):
                if preds[i].item() != labels[i].item():
                    misclassified.append((images[i], labels[i].item(), preds[i].item()))
                if len(misclassified) >= n:
                    return misclassified
    return misclassified


# Analyse misclassifications from both models
mis_cnn = collect_misclassified(basic_cnn,    test_dataset, test_loader, n=10)
mis_rn  = collect_misclassified(resnet_model, test_dataset, test_loader, n=10)
print(f'Misclassified – CNN: {len(mis_cnn)}   ResNet: {len(mis_rn)}')

In [ ]:
def show_misclassified(misclassified, model_name, inv_norm):
    n   = min(len(misclassified), 10)
    cols = 5
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.5))
    axes = axes.flat if rows > 1 else [axes] if cols == 1 else axes.flat

    for ax, (img_t, true_l, pred_l) in zip(axes, misclassified[:n]):
        img = inv_norm(img_t).permute(1, 2, 0).clamp(0, 1).numpy()
        ax.imshow(img)
        ax.set_title(
            f'True: {idx2class[true_l].split("_", 1)[-1]}\n'
            f'Pred: {idx2class[pred_l].split("_", 1)[-1]}',
            fontsize=7, color='red'
        )
        ax.axis('off')

    # hide unused subplots
    for ax in list(axes)[n:]:
        ax.axis('off')

    plt.suptitle(f'Misclassified Samples – {model_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


show_misclassified(mis_cnn, 'Basic CNN',  inv_norm)
show_misclassified(mis_rn,  'ResNet-18',  inv_norm)

### Misclassification Analysis

**Possible reasons for misclassification:**

1. **Visual similarity between adjacent AQI classes** – The difference between *Moderate* and
   *Unhealthy for Sensitive Groups* is subtle; both appear as slightly hazy outdoor scenes with
   similar colour palettes. The model has no chemical sensor readings, so it must rely entirely on
   visual cues that are inherently ambiguous between neighbouring classes.

2. **Lighting and time-of-day variation** – Images taken at dusk or under artificial lighting may
   appear hazier than their actual AQI warrants, causing the model to over-predict higher pollution
   classes.

3. **Scene diversity** – AQI is a property of the air, not of a specific landmark. The same AQI
   level can appear across urban highways, parks, or rooftops, each with very different background
   textures. This makes it harder for the CNN to generalise from training scenes to unseen locations.

4. **Low spatial resolution of haze cues** – At 224 × 224 the model may not capture fine-grained
   atmospheric detail (skyline clarity, shadow sharpness) that a human observer would use.

5. **Class boundary ambiguity in labelling** – The AQI boundaries (e.g., 100 vs 101) are
   technically discrete but visually continuous; images right on the boundary can legitimately
   belong to either adjacent class.

## Summary – Performance Comparison

In [ ]:
summary = pd.DataFrame({
    'Model':     ['Basic CNN (scratch)', 'ResNet-18 (transfer)'],
    'Accuracy':  [round(acc_cnn, 4),  round(acc_rn, 4)],
    'Precision': [round(prec_cnn, 4), round(prec_rn, 4)],
    'Recall':    [round(rec_cnn, 4),  round(rec_rn, 4)],
    'F1-Score':  [round(f1_cnn, 4),   round(f1_rn, 4)],
})
print(summary.to_string(index=False))